# Part D: Quantitative Mechanical Feature Extraction

## Overview
This notebook extracts morphological and mechanical features from the 16-bit labelled masks produced in **Part C** and performs comparative analysis across three genotypes: **Native**, **OE (Over-expressed)**, and **KD (Knock-down)** of Cytoskeletal Protein X.

### Pipeline Stages
1. **Per-condition area sampling** — establishes a 90% CI filter for each genotype independently.
2. **Feature extraction loop** — applies area + bounding-box aspect ratio filters, then computes all shape/deformation metrics.
3. **Export** — saves a consolidated CSV for downstream statistical analysis.
4. **Visualisation** — area distributions, deformation scatter, and boxplot comparison.

### Quality-Control Filters
- **Area CI filter (per condition):** 90% CI using mean ± 1.645 × SD, clamped to a minimum of 1 px. Computed *per genotype* because OE/KD cells may be intrinsically larger or smaller than Native — a global CI would incorrectly penalise legitimate biological size differences.
- **Bounding-box aspect ratio filter:** Excludes highly elongated objects (doublets, debris, red blood cells). Valid range: 0.5 – 2.0.

## 1. Imports & Path Configuration

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from skimage.measure import regionprops
from skimage.io import imread

# ── Paths ─────────────────────────────────────────────────────────────────
base_dir    = os.path.dirname(os.getcwd())
mask_root   = os.path.join(base_dir, "data", "processed", "masks")

output_csv  = os.path.join(base_dir, "data", "results", "CytoskeletalProteinX_Mechanical_Features.csv")

os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# Genotype folders expected inside mask_root/
conditions = ["Native_cells", "OE_cells", "KD_cells"]

print(f"Mask root : {mask_root}")
print(f"Output CSV: {output_csv}")

## 2. Per-Condition Area Sampling & 90% CI Calculation

**Why per-condition?** Applying a single global CI across all genotypes would pool cells of inherently different sizes. Because Cytoskeletal Protein X modifies cortical stiffness and morphology, OE and KD cells may be systematically larger or smaller than Native. A per-condition CI ensures the filter is calibrated to each population's own distribution.

In [ ]:
ci_bounds        = {}   # {condition: (lower_bound, upper_bound)}
condition_areas  = {}   # {condition: [area, ...]}  — kept for plotting

for condition in conditions:
    cond_path = os.path.join(mask_root, condition)
    if not os.path.exists(cond_path):
        print(f"WARNING: folder not found — {cond_path}")
        continue

    all_areas = []
    for mask_file in sorted(os.listdir(cond_path)):
        if not mask_file.endswith('.png'):   
            continue
        img = imread(os.path.join(cond_path, mask_file))

        #   Calling label(img>0) discards those labels and re-labels unnecessarily.
        #   regionprops() accepts the labelled mask directly (background = 0).
        for prop in regionprops(img):
            all_areas.append(prop.area)

    mean_a = np.mean(all_areas)
    std_a  = np.std(all_areas)
    lo = mean_a - 1.645 * std_a
    hi = mean_a + 1.645 * std_a

    lo = max(lo, 1)

    ci_bounds[condition]       = (lo, hi)
    condition_areas[condition] = all_areas
    print(f"{condition:15s}  n={len(all_areas):>6}  mean={mean_a:.1f}  std={std_a:.1f}  "
          f"90% CI=[{lo:.1f}, {hi:.1f}]")

## 3. Area Distribution — Visual QC

Each genotype's area histogram is plotted with its own 90% CI bounds. This confirms that the per-condition filter is correctly calibrated before feature extraction runs.

In [ ]:

colors = {"Native_cells": "steelblue", "OE_cells": "tomato", "KD_cells": "seagreen"}

fig, axes = plt.subplots(1, len(conditions), figsize=(14, 4), sharey=False)

for ax, condition in zip(axes, conditions):
    if condition not in condition_areas:
        ax.set_visible(False)
        continue
    lo, hi = ci_bounds[condition]
    sns.histplot(condition_areas[condition], kde=True, bins=30,
                 color=colors[condition], alpha=0.6, ax=ax)
    ax.axvline(lo, color='red', linestyle='--', linewidth=1.2, label=f'90% CI lower ({lo:.0f})')
    ax.axvline(hi, color='red', linestyle='--', linewidth=1.2, label=f'90% CI upper ({hi:.0f})')
    ax.set_title(condition.replace('_', ' '), fontsize=12)
    ax.set_xlabel("Cell Area (px)", fontsize=10)
    ax.set_ylabel("Frequency", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("Per-Condition Cell Area Distribution with 90% CI Bounds", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature Extraction Loop

For each valid cell (passing both QC filters), the following features are extracted:

**Shape:** Area, Perimeter, Circularity, Solidity, Eccentricity, Orientation  
**Deformation metrics:** Deformation (*D*), AspectRatio (major/minor), Roundness  
**Geometry:** MajorAxisLength, MinorAxisLength, BoundingBox, BoundingBox_AspectRatio, Centroid

In [ ]:
global_results = []

for condition in conditions:
    cond_path = os.path.join(mask_root, condition)
    if condition not in ci_bounds:
        continue
    lo, hi = ci_bounds[condition]

    mask_files = sorted([f for f in os.listdir(cond_path) if f.endswith('.png')])
    print(f"Processing {condition}: {len(mask_files)} frames ...")

    for frame_number, mask_file in enumerate(mask_files, start=1):
        img = imread(os.path.join(cond_path, mask_file))

        for prop in regionprops(img):

            # ── Filter 1: per-condition area CI ───────────────────────────
            area = prop.area
            if area < lo or area > hi:
                continue

            # ── Filter 2: bounding-box aspect ratio ───────────────────────
            min_row, min_col, max_row, max_col = prop.bbox
            bbox_h = max_row - min_row
            bbox_w = max_col - min_col
            bbox_ar = bbox_w / bbox_h if bbox_h > 0 else 0
            if bbox_ar < 0.5 or bbox_ar > 2.0:
                continue

            # ── Feature computation ────────────────────────────────────────

            peri  = prop.perimeter if prop.perimeter > 0 else 1
            major = prop.major_axis_length if prop.major_axis_length > 0 else 1
            minor = prop.minor_axis_length if prop.minor_axis_length > 0 else 1

            # Deformation  D = 1 - 2√(π·A) / P   (RT-DC literature definition)
            deformation  = 1 - (2 * math.sqrt(math.pi * area)) / peri
            # Circularity  C = 4π·A / P²          (1 = perfect circle)
            circularity  = (4 * np.pi * area) / (peri ** 2)
            # Aspect ratio AR = major / minor
            aspect_ratio = major / minor
            # Roundness    R = 4·A / (π·major²)
            roundness    = 4 * area / (math.pi * (major ** 2))

            cx, cy = prop.centroid

            global_results.append({
                "Condition":            condition,
                "Frame":                frame_number,
                "Cell_ID":              prop.label,
                "Area":                 area,
                "Perimeter":            peri,
                "Deformation":          deformation,
                "Circularity":          circularity,
                "Solidity":             prop.solidity,
                "AspectRatio":          aspect_ratio,
                "Roundness":            roundness,
                "Eccentricity":         prop.eccentricity,
                "Orientation":          prop.orientation,
                "MajorAxisLength":      major,
                "MinorAxisLength":      minor,
                "BoundingBox":          (min_row, min_col, max_row, max_col),
                "BoundingBox_AR":       bbox_ar,
                "Centroid_X":           cy,
                "Centroid_Y":           cx,
            })

df = pd.DataFrame(global_results)
df.to_csv(output_csv, index=False)
print(f"\nExtraction complete. Total valid cells: {len(df)}")
print(f"Saved to: {output_csv}")
df.groupby('Condition')[['Area','Deformation','Circularity','Solidity']].describe().round(4)

## 5. Comparative Visualisation

In [ ]:
palette = {"Native_cells": "steelblue", "OE_cells": "tomato", "KD_cells": "seagreen"}

# ── Plot 1: Area vs Deformation scatter ───────────────────────────────────
plt.figure(figsize=(10, 6))
for cond, grp in df.groupby('Condition'):
    plt.scatter(grp['Area'], grp['Deformation'],
                label=cond.replace('_',' '), alpha=0.4, s=10, color=palette[cond])
plt.xlabel("Cell Area (px)", fontsize=13)
plt.ylabel("Deformation (D)", fontsize=13)

plt.title("Cell Area vs Deformation — Cytoskeletal Protein X Genotypes", fontsize=13)
plt.legend(title='Condition')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(output_csv), 'scatter_area_deformation.png'), dpi=300)
plt.show()

# ── Plot 2: Deformation boxplot ────────────────────────────────────────────
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Condition', y='Deformation',
            palette=palette, order=conditions)
plt.xlabel("Genotype", fontsize=13)
plt.ylabel("Deformation (D)", fontsize=13)

plt.title("Mechanical Phenotype Comparison — Cytoskeletal Protein X", fontsize=13)
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(output_csv), 'boxplot_deformation.png'), dpi=300)
plt.show()

# ── Plot 3: Circularity boxplot ────────────────────────────────────────────
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Condition', y='Circularity',
            palette=palette, order=conditions)
plt.xlabel("Genotype", fontsize=13)
plt.ylabel("Circularity", fontsize=13)
plt.title("Circularity Comparison — Cytoskeletal Protein X", fontsize=13)
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(output_csv), 'boxplot_circularity.png'), dpi=300)
plt.show()